# Tutorial on how to do a complete experiment using Lexos

## Gabe's brainstorm
Flow of experiment:
1 - Import using Loader from IO module
2 - Normalize using util.normalize (import util)
3 - Scrub using Scrubber module (Scrubber() instance)
4 - Tokenize using Tokenizer () instance
5 - If Corups is available, add Records() instances of the texts to Corpus
6 - Create DTM() instance

Attachement tools for experiments:
- Cluster
- Classification
- Topic modelling (no clue on this one)
- Milestones
- Rolling Windows
- Decision tree

This end‑to‑end tutorial walks you through a realistic Lexos text analysis experiment:

1. Project & data setup
2. Loading raw text files
3. Normalizing & scrubbing (cleaning) text
4. Tokenization strategies
5. Building / interacting with a Corpus (records, metadata, stats)
6. Exploring term & token statistics
7. Creating document-term data for downstream modeling
8. Clustering / similarity & exporting analysis fingerprints
9. Reproducibility & validating previous analyses
10. Putting it all together in a reusable workflow function

We'll incrementally build from raw text → cleaned documents → structured stats → analysis outputs.

Assumptions / prerequisites:
- lexos is installed in this environment (editable install from repo or `pip install lexos` if published)
- spaCy small English model available (install via: `python -m spacy download en_core_web_sm` if needed)
- You are running this inside the project root so relative imports work.

If something fails due to an unavailable component (e.g., experimental Corpus APIs), the notebook shows fallbacks using working lower‑level pieces (Record + utility functions).

Let’s dive in.


## 1. Environment & Data Setup

In a real project you typically start with a directory of raw text sources. For this tutorial we'll:
- Create a small in‑memory collection of sample documents (you can replace with file loading later)
- Show how to normalize encodings / newlines
- Prepare a simple metadata table

If you have a folder of `.txt` files you can adapt the optional code in the next code cell.


In [ ]:
# Core imports for this tutorial
from __future__ import annotations

import os, textwrap, json, uuid, math, statistics
from pathlib import Path
from collections import Counter

import pandas as pd

# Lexos imports (robust to partial availability)
from lexos import util
from lexos.scrubber import scrub
from lexos.scrubber.scrubber import Scrubber
from lexos.scrubber.pipeline import make_pipeline, pipe
from lexos.scrubber.normalize import lower_case
from lexos.scrubber.remove import digits, punctuation
from lexos.tokenizer import Tokenizer, SliceTokenizer, WhitespaceTokenizer
from lexos.corpus.record import Record
from lexos.corpus.utils import RecordsDict, LexosModelCache

# Some components (Corpus, CorpusStats) may be experimental / unavailable
try:
    from lexos.corpus.corpus import Corpus
    CORPUS_AVAILABLE = True
except Exception as e:
    CORPUS_AVAILABLE = False
    CORPUS_IMPORT_ERROR = repr(e)

try:
    from lexos.corpus.corpus_stats import CorpusStats
    CORPUS_STATS_AVAILABLE = True
except Exception as e:
    CORPUS_STATS_AVAILABLE = False
    CORPUS_STATS_IMPORT_ERROR = repr(e)

print(f"Corpus available: {CORPUS_AVAILABLE}")
print(f"CorpusStats available: {CORPUS_STATS_AVAILABLE}")

# Sample raw texts (simulate project input)
RAW_TEXTS = {
    "orwell_1.txt": "It was a bright cold day in April, and the clocks were striking thirteen.",
    "orwell_2.txt": "Winston Smith, his chin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through the glass doors.",
    "dickens_1.txt": "It was the best of times, it was the worst of times, it was the age of wisdom.",
    "dickens_2.txt": "It was the age of foolishness, it was the epoch of belief, it was the epoch of incredulity.",
}

# Minimal metadata (author + synthetic genre)
META = [
    {"filename": fn, "author": ("Orwell" if "orwell" in fn else "Dickens"), "genre": "novel", "year_group": 1 if i < 2 else 2}
    for i, fn in enumerate(RAW_TEXTS.keys())
]

pd.DataFrame(META)  # display

## 2. Normalizing Raw Texts
Lexos provides utilities to normalize encodings/newlines. Here we simulate reading raw bytes / mixed newlines and produce clean UTF‑8 strings. In real usage you'd do something like:

```
from lexos import util
raw_bytes = Path('data/file1.txt').read_bytes()
clean_text = util.normalize(raw_bytes)
```

We'll batch normalize all sample texts now.

In [ ]:
# Normalize all texts (simulate reading bytes)
normalized_texts = {
    fn: util.normalize(txt if isinstance(txt, str) else txt.encode())
    for fn, txt in RAW_TEXTS.items()
}

# Show a preview (first 60 chars) to verify
for k, v in normalized_texts.items():
    print(f"{k}: {v[:60]}...")

# (Optional) Convert to DataFrame for inspection
raw_df = pd.DataFrame([
    {"filename": fn, "raw_length": len(RAW_TEXTS[fn]), "normalized_length": len(txt)}
    for fn, txt in normalized_texts.items()
])
raw_df

## 3. Scrubbing / Cleaning Text

Lexos's Scrubber lets you build ordered pipelines of transformations (normalize casing, remove digits/punctuation, etc.). You can:
- Use `scrub(text, pipeline)` for a quick, ad‑hoc pipeline
- Or build a reusable `Scrubber` object, add components, then apply to many texts (with optional disabling / dynamic config).

Below we demonstrate both approaches.

In [ ]:
# --- Quick ad-hoc scrubbing with `scrub` ---
basic_pipeline = ["lower_case", (digits, {"only": []}), (punctuation, {"only": [","]})]
example = list(RAW_TEXTS.values())[0]
print("Original:", example)
print("Scrubbed:", scrub(example, basic_pipeline))

# --- Reusable Scrubber object ---
s = Scrubber()
s.add_pipe(["lower_case", ("digits", {"only": []})])
# Dynamically extend pipeline
s.add_pipe(("punctuation", {"only": [","]}), last=True)

multi_scrubbed = list(s.pipe([t for t in normalized_texts.values()]))
print(f"Scrubbed {len(multi_scrubbed)} docs. Preview first: \n{multi_scrubbed[0]}")

SCRUBBED_TEXTS = {fn: txt for fn, txt in zip(normalized_texts.keys(), multi_scrubbed)}

## 4. Tokenization Strategies
Lexos wraps spaCy for flexible tokenization and also provides lighter tokenizers for special cases.

We will:
- Create a standard `Tokenizer`
- Add/remove stop words
- Demonstrate whitespace + slice tokenizers for alternative analyses.


In [ ]:
# Create main tokenizer (defaults to small English model or blank fallback)
try:
    tokenizer = Tokenizer()
except Exception as e:
    print("Tokenizer model load issue, falling back to blank 'en':", e)
    tokenizer = Tokenizer(model="en")

# Add a custom extension & stopwords
tokenizer.add_extension("is_caps", default=False)
custom_stops = ["the", "was", "of", "it"]
tokenizer.add_stopwords(custom_stops)

TOKENIZED_DOCS = {}
for fn, txt in SCRUBBED_TEXTS.items():
    doc = tokenizer(txt)
    # Mark tokens that were uppercase originally (using original raw) just as demo
    for token in doc:
        token._.is_caps = token.text.isupper()
    TOKENIZED_DOCS[fn] = doc

# Show a sample token list
[(t.text, t._.is_caps) for t in list(TOKENIZED_DOCS.values())[0][:12]]

### Alternative tokenizers
Slice and whitespace tokenizers can create different views over the text (character windows or raw whitespace tokens).

In [ ]:
slice_tok = SliceTokenizer(n=5, drop_ws=True)
ws_tok = WhitespaceTokenizer()

print("Slice tokenizer sample:", slice_tok(list(SCRUBBED_TEXTS.values())[0])[:6])
print("Whitespace tokenizer sample:", list(ws_tok(list(SCRUBBED_TEXTS.values())[0].split(" ")))[:10])

## 5. Building a Corpus (or Simulating One)
If `Corpus` is available we'll use it; otherwise we'll simulate a lightweight corpus structure with `Record` + `RecordsDict` as in the test suite.


In [ ]:
if CORPUS_AVAILABLE:
    corpus = Corpus(name="DemoCorpus")
    # Add tokenized docs as Records directly
    for fn, doc in TOKENIZED_DOCS.items():
        corpus.add(content=doc, name=fn.replace('.txt',''))
    print("Corpus docs:", corpus.num_docs)
else:
    print("Corpus unavailable, simulating with RecordsDict")
    corpus_sim = {
        'records': RecordsDict(),
        'stats': {'total_docs': 0, 'active_docs': 0}
    }
    for fn, doc in TOKENIZED_DOCS.items():
        record = Record(id=uuid.uuid4(), name=fn.replace('.txt',''), content=doc, model='en_core_web_sm', is_active=True)
        corpus_sim['records'][str(record.id)] = record
        corpus_sim['stats']['total_docs'] += 1
        corpus_sim['stats']['active_docs'] += 1
    print("Simulated records:", corpus_sim['stats'])

## 6. Exploring Token & Term Statistics
If `CorpusStats` is available we can compute aggregated stats. Otherwise we manually aggregate from tokenized docs.


## 7. Creating a Document-Term Matrix (DTM)
Lexos includes a DTM module (see tests) but if it's not available we can construct a simple wide matrix with pandas for modeling.


In [ ]:
# Simple DTM builder (unweighted counts)

def build_dtm(tokenized_docs: dict[str, 'Doc'], min_df=1, max_df=1.0):
    term_doc_counts = Counter()
    per_doc_terms: dict[str, Counter] = {}
    for name, doc in tokenized_docs.items():
        terms = Counter([t.text for t in doc if not t.is_space])
        per_doc_terms[name] = terms
        for term in terms.keys():
            term_doc_counts[term] += 1
    n_docs = len(per_doc_terms)
    filtered_terms = {
        term for term, df in term_doc_counts.items()
        if df >= min_df and df / n_docs <= max_df
    }
    rows = []
    for name, counts in per_doc_terms.items():
        row = {term: counts.get(term, 0) for term in filtered_terms}
        row['__doc__'] = name
        rows.append(row)
    dtm = pd.DataFrame(rows).set_index('__doc__').sort_index(axis=1)
    return dtm

DTM = build_dtm(TOKENIZED_DOCS, min_df=1, max_df=1.0)
DTM.head()

## 8. Simple Similarity / Clustering Example
We'll compute a cosine similarity matrix and perform a basic hierarchical clustering (scipy) if available. This stands in for more advanced Lexos analysis modules (cluster, topic modeling, etc.).

In [ ]:
import numpy as np
try:
    from scipy.spatial.distance import pdist, squareform
    from scipy.cluster.hierarchy import linkage, dendrogram
    SCIPY = True
except Exception:
    SCIPY = False

# Cosine similarity
norms = np.linalg.norm(DTM.values, axis=1, keepdims=True)
normalized = DTM.values / np.where(norms==0, 1, norms)
sim_matrix = normalized @ normalized.T
sim_df = pd.DataFrame(sim_matrix, index=DTM.index, columns=DTM.index)
print("Cosine similarity matrix:")
display(sim_df.round(3))

if SCIPY:
    dist_vec = pdist(normalized, metric='cosine')
    link = linkage(dist_vec, method='average')
    # For notebook visualization you could call dendrogram(link), but we keep side‑effect minimal
    print("Hierarchical clustering linkage matrix (first rows):")
    print(link[:5])
else:
    print("scipy not installed; skipping dendrogram example.")

# Prepare analysis results payload
analysis_results = {
    'similarity_matrix': sim_df.round(4).to_dict(),
    'method': 'cosine',
    'docs': list(DTM.index),
}

if CORPUS_AVAILABLE:
    corpus.import_analysis_results(module_name='similarity_demo', results_data=analysis_results, version='0.1.0', overwrite=True)
    print("Stored analysis results in corpus.")

## 9. Reproducibility & Fingerprinting
Using the corpus fingerprint + validation methods you can ensure that an analysis result still matches the current corpus state.


In [ ]:
if CORPUS_AVAILABLE:
    fp = corpus.export_statistical_fingerprint()
    print("Fingerprint keys:", list(fp.keys()))
    compat = corpus.validate_analysis_compatibility('similarity_demo')
    print("Compatibility result:")
    display(compat)
else:
    print("Corpus not available; skipping fingerprint demonstration.")

## 10. Wrapping It All Up: Reusable Experiment Function
We now package the end‑to‑end workflow into a single function you can adapt.


In [ ]:
def run_lexos_experiment(text_map: dict[str, str], *, lowercase=True, remove_digits=True, remove_punct=True,
                          min_df=1, max_df=1.0) -> dict:
    # 1. Normalize
    norm = {k: util.normalize(v) for k,v in text_map.items()}

    # 2. Scrub pipeline config
    pipeline = []
    if lowercase: pipeline.append("lower_case")
    if remove_digits: pipeline.append("digits")
    if remove_punct: pipeline.append("punctuation")
    scrubbed = {k: scrub(v, pipeline) for k,v in norm.items()}

    # 3. Tokenize
    try:
        tok = Tokenizer()
    except Exception:
        tok = Tokenizer(model="en")
    docs = {k: tok(v) for k,v in scrubbed.items()}

    # 4. Build DTM
    dtm = build_dtm(docs, min_df=min_df, max_df=max_df)

    # 5. Similarity
    norms = np.linalg.norm(dtm.values, axis=1, keepdims=True)
    normalized = dtm.values / np.where(norms==0, 1, norms)
    sim = normalized @ normalized.T
    sim_df = pd.DataFrame(sim, index=dtm.index, columns=dtm.index)

    return {
        'normalized_texts': norm,
        'scrubbed_texts': scrubbed,
        'docs': docs,
        'dtm': dtm,
        'similarity': sim_df
    }

experiment_out = run_lexos_experiment(RAW_TEXTS)
experiment_out['dtm'].head()

### Next Steps
- Swap the in‑memory `RAW_TEXTS` with file ingestion (use `Path.glob` + `read_bytes` + `util.normalize`)
- Experiment with additional scrubber components (see `scrubber_components.registry`)
- Integrate visualization modules and topic modeling once stable
- Persist corpus using `corpus.save()` and reload with `corpus.load()` when Corpus becomes fully available

This concludes the full Lexos flow tutorial.

In [ ]:
def manual_term_stats(tokenized_docs: dict[str, 'Doc']) -> pd.DataFrame:
    rows = []
    for name, doc in tokenized_docs.items():
        tokens = [t.text for t in doc if not t.is_space]
        terms = Counter(tokens)
        rows.append({
            'doc': name,
            'num_tokens': len(tokens),
            'num_terms': len(terms),
            'vocab_density': len(terms)/len(tokens) if tokens else 0,
            'top_terms': ', '.join([f"{w}:{c}" for w,c in terms.most_common(5)])
        })
    return pd.DataFrame(rows)

if CORPUS_AVAILABLE and CORPUS_STATS_AVAILABLE:
    stats = corpus.get_stats()
    # Hypothetical: show DataFrame if attribute exists
    if hasattr(stats, 'doc_stats_df'):
        display(stats.doc_stats_df.head())
    else:
        print("CorpusStats available but doc_stats_df attribute missing; using manual fallback.")
        display(manual_term_stats(TOKENIZED_DOCS))
else:
    display(manual_term_stats(TOKENIZED_DOCS))